### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "survey_points.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp01/survey_points.csv"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp01" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP01 — First GeoPython Workflow

Load → Inspect → Understand → Filter → Map → Validate → Export

Σε κάθε στάδιο γράψε τι περιμένεις να αλλάξει και τι θα ελέγξεις. Το CSV είναι αρχικά DataFrame· μετά τη δημιουργία geometry θα γίνει GeoDataFrame. Μην προχωράς σε επόμενο στάδιο αν δεν μπορείς να ερμηνεύσεις το output του προηγούμενου.

## 1. Load — Φόρτωση δεδομένων

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path


In [ ]:
csv_path = DATA_ROOT / "gp01/survey_points.csv"
points = pd.read_csv(csv_path)
points.head()


## 2. Inspect — Πρώτη επιθεώρηση

Εμφάνισε το μέγεθος, τις στήλες, τους τύπους και τις κενές τιμές. Το `shape` είναι γραμμές/στήλες, τα `dtypes` επηρεάζουν τις πράξεις και το `isna().sum()` δείχνει πού χρειάζεται προσοχή.

In [ ]:
# Συμπλήρωσε τους ελέγχους
# print(points.shape)
# print(points.columns)
# print(points.dtypes)
# print(points.isna().sum())


## 3. Understand — Δημιουργία γεωμετρίας

In [ ]:
geometry = [Point(xy) for xy in zip(points["longitude"], points["latitude"]) ]
gdf = gpd.GeoDataFrame(points, geometry=geometry, crs="EPSG:4326")
gdf.head()


## 4. Filter — Επιλογή εγγραφών

In [ ]:
# Κράτησε checked σημεία με υψόμετρο πάνω από 70 m
selected = gdf[
    (gdf["elevation_m"] > 70) &
    (gdf["status"] == "checked")
] .copy()
selected


## 5. Map — Χάρτης

In [ ]:
ax = gdf.plot(figsize=(8, 6), color="lightgray")
selected.plot(ax=ax, color="red", markersize=60)


## 6. Validate — GeoPython QA checks

In [ ]:
# Συμπλήρωσε ή εκτέλεσε τους QA checks
# assert len(gdf) > 0
# assert gdf.geometry.notna().all()
# assert gdf.geometry.geom_type.eq("Point").all()
# assert gdf.crs is not None
# assert (selected["elevation_m"] > 70).all()


## 7. Export — Αποθήκευση αποτελέσματος

In [ ]:
output_dir = Path("outputs/gp01")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "high_checked_points.geojson"
# selected.to_file(output_path, driver="GeoJSON")


## Mini Challenge

Βρες τα `water` σημεία με υψόμετρο μικρότερο από 65 m, κάνε QA και αποθήκευσέ τα ως `water_low_points.geojson`.

In [ ]:
# Γράψε εδώ τη λύση σου
